# 10 — Unique vs Total CDR3 Reads

Methodological supplement comparing depth-adjusted unique CDR3 clonotypes
(primary metric) against depth-adjusted total CDR3 reads — T-cell only.

**Panels:**
- A: Correlation between unique and total depth-adjusted scores (T-cell)
- B: RNA gene set validation (T-cell) — unique vs total
- C: ImmuneLENS validation (T-cell loci) — unique vs total
- D: Forest plots — OS and BOR effect estimates, shape by read type (T-cell)

**Prerequisites:** `4_base.ipynb` and `5_models.ipynb` must have been run.


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

library(survival)
library(broom)
library(patchwork)
library(ggpubr)

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# ready.rds    — all patients (used for RNA / ImmuneLENS validation)
# cpi_ready    — ICI-treated with response data (used for forest models)
ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds"))
cpi   <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds"))

# Derive per-locus RNA score (not stored in ready.rds; computed same way as 4_cider_validation.ipynb)
ready <- ready |>
  dplyr::mutate(rna = tcell_tpm)

# Restrict to T-cell (this project is T-cell-focused)
GP_TB <- c("T-cell")

if (!"cider_total_depth_adj" %in% colnames(ready)) {
  warning("cider_total_depth_adj missing — computing inline. Re-run 4_base.ipynb for full pipeline consistency.")
  ready <- ready |> dplyr::mutate(cider_total_depth_adj = (cider_total / sampleMedianDepth) * SCORE_SCALE)
  cpi   <- cpi   |> dplyr::mutate(cider_total_depth_adj = (cider_total / sampleMedianDepth) * SCORE_SCALE)
}

# Panel A — Unique vs Total depth-adjusted score

In [ ]:
n_wgs <- ready |>
  dplyr::filter(!is.na(cider_score_depth_adj), !is.na(cider_total_depth_adj),
                gp %in% GP_TB) |>
  dplyr::distinct(sampleId) |>
  nrow()

scatter_data <- ready |>
  dplyr::filter(!is.na(cider_score_depth_adj), !is.na(cider_total_depth_adj),
                gp %in% GP_TB) |>
  dplyr::mutate(
    # Compute density in log10(x+1) space to match visual axis transformation
    pt_alpha = density_alpha(
      cider_score_depth_adj, cider_total_depth_adj,
      tx = function(v) log10(v + 1),
      ty = function(v) log10(v + 1)
    )
  )

p_scatter <- scatter_data |>
  ggplot(aes(x = cider_score_depth_adj, y = cider_total_depth_adj)) +
  geom_point(aes(alpha = pt_alpha), size = 0.6, colour = COL_NEGATIVE) +
  scale_alpha_identity(guide = "none") +
  # "grey20" (near-black), not COL_NEGATIVE -- same convention as Figure 5A's
  # regression line (5_cider_vs_tmb.ipynb's tmb_vs_tcell: "Dark charcoal grey
  # rather than ggplot's default blue -- keeps the fitted line visually
  # neutral against the categorical palette used elsewhere in the paper").
  # Deliberately darker than the grey points themselves so the line reads as
  # a distinct fitted trend, not just another point layer.
  geom_smooth(method = "lm", colour = "grey20", se = FALSE, linewidth = 0.8) +
  stat_cor(
    aes(label = after_stat(r.label)),
    method = "pearson", size = 2.8,
    label.x.npc = 0.05, label.y = Inf, vjust = 1.3
  ) +
  coord_cartesian(clip = "off") +
  scale_x_continuous(trans = log10p1_trans, expand = expansion(mult = c(0.02, 0.05)), breaks = c(0, 1, 10, 100)) +
  scale_y_continuous(trans = log10p1_trans, expand = expansion(mult = c(0.02, 0.01)), breaks = c(0, 1, 10, 100)) +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    x        = "CIDER Score (Unique Reads)",
    y        = "CIDER Score (Total Reads)",
    title    = "Unique vs Total T-cell Reads",
    subtitle = paste0("N = ", format(n_wgs, big.mark = ","), " samples \u00b7 Hartwig WGS")
  )

set_plot_dims(6, 3.5); p_scatter

# Panel B — Combined validation summary: RNA + ImmuneLENS


In [ ]:
# ── RNA correlation summary (r per locus × metric_type) ─────────────────────
rna_long <- ready |>
  dplyr::filter(!is.na(rna), gp %in% GP_TB) |>
  tidyr::pivot_longer(
    cols      = c(cider_score_depth_adj, cider_total_depth_adj),
    names_to  = "metric_raw",
    values_to = "cider"
  ) |>
  dplyr::mutate(
    metric_type = dplyr::case_when(
      metric_raw == "cider_score_depth_adj" ~ "Unique Reads",
      metric_raw == "cider_total_depth_adj" ~ "Total Reads"
    ),
    metric_type = factor(metric_type, levels = c("Unique Reads", "Total Reads")),
    gp          = factor(gp, levels = "T-cell")
  )

rna_cor_summary <- rna_long |>
  dplyr::group_by(gp, metric_type) |>
  dplyr::summarise(
    r      = cor(cider, rna, use = "pairwise.complete.obs", method = "pearson"),
    method = "RNA gene set",
    .groups = "drop"
  )


# ── ImmuneLENS r-value summary + combine with RNA ───────────────────────────


In [ ]:
IL_GP        <- c("T-cell", "T-cell (TRA/TRD)", "T-cell (TRB)", "T-cell (TRG)")
IL_GP_LABELS <- c("T-cell", "TRA/TRD", "TRB", "TRG")

il_long <- cpi |>
  dplyr::filter(!is.na(immuneLENS_fraction), gp %in% IL_GP) |>
  tidyr::pivot_longer(
    cols      = c(cider_score_depth_adj, cider_total_depth_adj),
    names_to  = "metric_raw",
    values_to = "cider"
  ) |>
  dplyr::mutate(
    metric_type = dplyr::case_when(
      metric_raw == "cider_score_depth_adj" ~ "Unique Reads",
      metric_raw == "cider_total_depth_adj" ~ "Total Reads"
    ),
    metric_type = factor(metric_type, levels = c("Unique Reads", "Total Reads")),
    gp          = factor(gp, levels = IL_GP, labels = IL_GP_LABELS)
  )

n_il  <- il_long  |> dplyr::distinct(sampleId) |> nrow()
n_rna <- rna_long |> dplyr::distinct(sampleId) |> nrow()

# ── Pearson r per row × metric_type ──────────────────────────────────────────
il_cor <- il_long |>
  dplyr::group_by(gp, metric_type) |>
  dplyr::summarise(r = cor(cider, immuneLENS_fraction,
                           use = "pairwise.complete.obs", method = "pearson"),
                   .groups = "drop") |>
  dplyr::mutate(label = paste0("ImmuneLENS ", gp))

rna_cor <- rna_long |>
  dplyr::group_by(metric_type) |>
  dplyr::summarise(r = cor(cider, rna,
                           use = "pairwise.complete.obs", method = "pearson"),
                   .groups = "drop") |>
  dplyr::mutate(label = "RNA T-cell")

# Combined label ordering (top to bottom)
label_levels <- rev(c(
  "RNA T-cell",
  "ImmuneLENS T-cell",
  "ImmuneLENS TRA/TRD",
  "ImmuneLENS TRB",
  "ImmuneLENS TRG"
))

val_summary <- dplyr::bind_rows(rna_cor, il_cor) |>
  dplyr::mutate(label = factor(label, levels = label_levels))

p_val_summary <- val_summary |>
  ggplot(aes(x = r, y = label, colour = metric_type, group = label)) +
  geom_line(colour = COL_NEGATIVE, linewidth = 0.8) +
  geom_point(size = 3.5) +
  # Unique vs Total is a read-type contrast, not Responder/High-Low-like;
  # this panel's own title frames "Unique Reads" as the notably stronger
  # metric, so it gets COL_ACCENT and "Total Reads" (baseline) gets COL_NEGATIVE.
  scale_colour_manual(
    values = c("Unique Reads" = COL_ACCENT, "Total Reads" = COL_NEGATIVE),
    name   = "Read type"
  ) +
  scale_x_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25),
                     expand = expansion(mult = c(0.02, 0.05))) +
  go_theme +
  theme(
    plot.subtitle   = element_text(hjust = 0.5),
    legend.position = "right"
    # panel.grid.major.y intentionally left at go_theme's default (none) --
    # a full-width horizontal gridline per row made it look like each row's
    # connector spanned the whole plot; the geom_line() above already draws
    # the actual connector, correctly spanning only Unique <-> Total Reads.
  ) +
  labs(
    x        = "Less agreement ← Pearson r → More agreement",
    y        = NULL,
    title    = "Unique T-cell Reads: Stronger External Validation",
    subtitle = paste0("RNA: N = ", format(n_rna, big.mark = ","),
                      " · ImmuneLENS: N = ", format(n_il, big.mark = ","),
                      " patients")
  )

set_plot_dims(6, 4); p_val_summary


# Panel C/D — Forest Plots: OS and BOR by Read Type

Compares CIDER Score effect size between Unique and Total reads across
cohorts and endpoints. Shape encodes read type; colour encodes endpoint.


In [ ]:
# ── Prepare long-format modelling data ────────────────────────────────────────
cpi_model <- cpi |>
  dplyr::filter(gp %in% GP_TB) |>
  dplyr::select(
    sampleId, gp, cohort_broad, cohort_broad_go,
    osDays, osEvent, pfsEventDays, pfsEvent,
    response, dcb,
    tmbPerMb,
    cider_score_depth_adj, cider_total_depth_adj
  ) |>
  tidyr::pivot_longer(
    cols      = c(cider_score_depth_adj, cider_total_depth_adj),
    names_to  = "metric_raw",
    values_to = "val"
  ) |>
  dplyr::mutate(
    metric_type = dplyr::case_when(
      metric_raw == "cider_score_depth_adj" ~ "Unique",
      metric_raw == "cider_total_depth_adj" ~ "Total"
    ),
    metric_type = factor(metric_type, levels = c("Total", "Unique")),
    # log2(x+1), unstandardized -- matches the convention used by every
    # other forest plot in the paper (0_prepare/5_models.ipynb,
    # 15_ici_specificity.ipynb, 20_ici_specificity_main.ipynb); previously
    # log1p() (natural log) + scale(), a different base AND standardized,
    # which wasn't comparable to the others.
    cider_log2 = log2(val + 1),
    scaled_tmb  = as.vector(scale(log2(tmbPerMb + 1)))
  ) |>
  tidyr::drop_na(val, tmbPerMb)

In [ ]:
# ── Fit OS, PFS, BOR, DCB models (unadjusted only) ───────────────────────────
fit_one <- function(df, model, formula, endpoint, covariates) {
  tryCatch({
    m    <- if (model == "coxph") coxph(formula, data = df) else glm(formula, data = df, family = binomial())
    tidy(m, conf.int = TRUE) |>
      dplyr::filter(term == "cider_log2") |>
      dplyr::mutate(endpoint = endpoint, covariates = covariates)
  }, error = function(e) NULL)
}

specs <- list(
  list(model = "coxph",    formula = Surv(osDays, osEvent)         ~ cider_log2, endpoint = "OS",  covariates = "None"),
  list(model = "coxph",    formula = Surv(pfsEventDays, pfsEvent)  ~ cider_log2, endpoint = "PFS", covariates = "None"),
  list(model = "logistic", formula = response                      ~ cider_log2, endpoint = "BOR", covariates = "None"),
  list(model = "logistic", formula = dcb                           ~ cider_log2, endpoint = "DCB", covariates = "None")
)

forest_data <- cpi_model |>
  dplyr::group_by(gp, cohort_broad, cohort_broad_go, metric_type) |>
  dplyr::group_modify(function(df, keys) {
    purrr::map_dfr(specs, function(s) {
      fit_one(df, s$model, s$formula, s$endpoint, s$covariates)
    })
  }) |>
  dplyr::ungroup() |>
  dplyr::mutate(
    gp              = factor(gp, levels = rev(GP_TB)),
    endpoint        = factor(endpoint, levels = c("OS", "PFS", "BOR", "DCB"))
  )

head(forest_data)

In [ ]:
# ── Forest plots: survival (OS+PFS) and response (BOR+DCB) ──────────────────
# Colour now encodes read type (Unique = COL_ACCENT, Total = COL_NEGATIVE),
# matching Panel B's val-summary colouring exactly, rather than endpoint --
# endpoint is already legible from the y-axis position/label within each
# facet, so colour was free to carry the read-type contrast instead. Shape
# still maps to the same read_type variable -- same "name"/"breaks" on both
# scales merges them into one combined colour+shape legend.
read_shapes <- c("Unique" = 16, "Total" = 17)
read_type_colors <- c("Unique" = COL_ACCENT, "Total" = COL_NEGATIVE)

n_forest <- cpi_model |> dplyr::distinct(sampleId) |> nrow()
forest_subtitle <- paste0("N = ", format(n_forest, big.mark = ","), " patients \u00b7 Hartwig ICI-Treated")

make_forest_panel <- function(data, endpoints, x_lab, title) {
  data |>
    dplyr::filter(endpoint %in% endpoints) |>
    # Place endpoints on y-axis (ordered top-to-bottom as supplied)
    dplyr::mutate(endpoint = factor(endpoint, levels = rev(endpoints))) |>
    ggplot(aes(
      x      = estimate,
      y      = endpoint,
      colour = metric_type,
      shape  = metric_type
    )) +
    geom_vline(xintercept = 0, linetype = "dashed", colour = COL_NEGATIVE) +
    geom_pointrange(
      aes(xmin = conf.low, xmax = conf.high),
      size = 0.5, fatten = 3,
      position = position_dodge(width = 0.6)
    ) +
    scale_colour_manual(values = read_type_colors, name = "Read Type",
                        breaks = c("Unique", "Total")) +
    scale_shape_manual(values = read_shapes, name = "Read Type",
                       breaks = c("Unique", "Total")) +
    scale_x_continuous(n.breaks = 3) +
    facet_grid(
      cols     = vars(cohort_broad_go),
      scales   = "free_x",
      labeller = labeller(cohort_broad_go = function(x) sub("ICI-Treated", "All ICI", x))
    ) +
    go_theme +
    theme(
      axis.text.y    = element_text(size = 8, face = "bold"),
      axis.text.x    = element_text(size = 7),
      strip.text     = element_text(size = 7),
      legend.position = "right",
      legend.text    = element_text(size = 7),
      legend.title   = element_text(size = 8, face = "bold"),
      plot.subtitle  = element_text(hjust = 0.5)
    ) +
    labs(x = x_lab, y = NULL, title = title, subtitle = forest_subtitle)
}

p_forest_survival <- make_forest_panel(
  forest_data, c("OS", "PFS"),
  x_lab = "Better Survival ← Log Hazard Ratio (95% CI) → Worse Survival",
  title = "Unique T-cell Reads Trend Toward Stronger Associations \u2014 Survival (OS and PFS)"
)
p_forest_response <- make_forest_panel(
  forest_data, c("BOR", "DCB"),
  x_lab = "Less Response ← Log Odds Ratio (95% CI) → More Response",
  title = "Unique T-cell Reads Trend Toward Stronger Associations \u2014 Response (BOR and DCB)"
)

set_plot_dims(10, 3); p_forest_response
set_plot_dims(10, 3); p_forest_survival


In [ ]:
# ── Assemble supplementary figure: Unique vs Total CDR3 Reads ────────────────
# A+B: scatter and validation summary (top row)
# C/D: forest plots — response and survival with shared legend
cd_panels <- (p_forest_response / p_forest_survival) +
  patchwork::plot_layout(guides = "collect") +
  figure_annotation(tag = FALSE, legend_pos = "right")

supplement_unique_vs_total <-
  (wrap_plots(p_scatter, p_val_summary, nrow = 1, widths = c(1, 1)) /
   cd_panels) +
  patchwork::plot_layout(heights = c(1.0, 1.2)) +
  figure_annotation()

set_plot_dims(10, 10); supplement_unique_vs_total


In [ ]:
# ── Save panels ───────────────────────────────────────────────────────────────
saveRDS(p_scatter,         paste0(CIDER_PROJ_DIR, "figs/unique_vs_total_scatter.rds"))
saveRDS(p_val_summary,     paste0(CIDER_PROJ_DIR, "figs/unique_vs_total_val_summary.rds"))
saveRDS(p_forest_response, paste0(CIDER_PROJ_DIR, "figs/unique_vs_total_forest_response.rds"))
saveRDS(p_forest_survival, paste0(CIDER_PROJ_DIR, "figs/unique_vs_total_forest_survival.rds"))
message("Panels saved.")
saveRDS(supplement_unique_vs_total,
        paste0(CIDER_PROJ_DIR, "figs/supplement_unique_vs_total.rds"))
